# 进阶教程（六）：最佳实践——模块化、配置、测试与性能

> 以本教程的 `rag_qa_project/` 为参照系，讲清 LLM 应用的工程化四件套。

## 本讲内容
1. 模块化代码结构（分层 + 依赖注入）
2. 配置管理（pydantic-settings）
3. 测试策略（三层金字塔，零额度回归）
4. 性能优化（并行 / batch / 上下文裁剪）

# 0. 环境准备与运行说明

**前置要求：**
- 根目录 `.env` 已配置 `DEEPSEEK_API_KEY`（本教程用真实 DeepSeek，无本地降级）
- 已安装：`langchain>=1.3`、`langgraph>=1.2`、`langchain-deepseek`、`python-dotenv`
- 使用本地 `bge-small-zh-v1.5` 嵌入的章节首次运行会下载模型（约 100MB，走 hf-mirror 镜像）

**运行说明：**
- 按 cell 顺序执行；除标注外，每个示例消耗少量 API 额度（单次 < 0.01 元量级）
- 本教程面向已学完 `langchain_tutorial/` 与 `langgraph_tutorial/` 基础篇的开发者
- 涉及导入路径的坑（如 `create_agent` 在 `langchain.agents`）已在 FAQ 中汇总

In [1]:

# ========== 0. 初始化（每个 notebook 第一格） ==========
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# HF 镜像必须先于任何 langchain/huggingface 导入设置（详见 rag_qa_project FAQ）
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

ROOT = Path.cwd().parent  # advanced_tutorial 的上一级 = 项目根目录
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("DEEPSEEK_API_KEY"), "请先在根目录 .env 配置 DEEPSEEK_API_KEY"

# 真实 LLM：DeepSeek（本教程要求真实模型）
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
print("模型就绪:", model.__class__.__name__)


模型就绪: ChatDeepSeek


## 1. 模块化代码结构

LLM 应用容易写成"一坨 notebook"，生产化的第一步是分层：

```
rag_qa_project/
├── config.py    # 配置层：所有魔法数字的唯一来源
├── ingest.py    # 数据层：知识库 -> 向量库（离线任务）
├── graph.py     # 编排层：LangGraph 工作流（核心逻辑）
├── run.py       # 接口层：CLI/API 薄壳（只做 IO 与参数解析）
└── tests/       # 测试层：Fake 注入，零额度回归
```

**三条纪律：**
1. **依赖注入**：`build_graph(model=None, retriever=None)` 把基础设施作为参数，
   测试塞 Fake、生产塞真货——`graph.py` 对"用什么模型"零假设
2. **节点纯函数**：输入 state → 输出增量 update，不碰全局变量（可回放、可测试）
3. **接口层最薄**：run.py 不写业务逻辑，换 Web API 时图和节点原封不动

## 2. 配置管理

规则：**代码里不允许出现裸常量**，一切进 `config.py`。
`pydantic-settings` 提供带类型校验的环境变量覆盖：

In [2]:

import os
from pydantic_settings import BaseSettings, SettingsConfigDict

class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="APP_", extra="ignore")

    model_name: str = "deepseek-chat"
    temperature: float = 0.1
    top_k: int = 4
    max_retries: int = 3

s = AppSettings()
print("默认配置:", s.model_name, s.temperature, s.top_k)

# 环境变量覆盖（生产发布不改代码）
os.environ["APP_TOP_K"] = "10"
os.environ["APP_TEMPERATURE"] = "0.5"
s2 = AppSettings()
print("覆盖后:", s2.temperature, s2.top_k)

# 类型错误会被拦截，而不是带病运行
os.environ["APP_TOP_K"] = "abc"
try:
    AppSettings()
except Exception as e:
    print("校验拦截:", type(e).__name__)

默认配置: deepseek-chat 0.1 4
覆盖后: 0.5 10
校验拦截: ValidationError


## 3. 测试策略：三层金字塔

| 层级 | 对象 | 手段 | 额度消耗 |
|---|---|---|---|
| 单元测试 | 节点函数、路由逻辑 | FakeChatModel 注入 | 0 |
| 集成测试 | 整图执行 | Fake 模型 + Fake 检索器跑全图 | 0 |
| 冒烟测试 | 端到端质量 | 真实 LLM 小样本 | 少量 |

`rag_qa_project/tests/test_graph.py` 已完整实现前两层。核心技巧：
**让 Fake 模型实现 `with_structured_output`**（真模型有、Fake 默认没有）：

In [3]:

# 单元测试示范：验证路由逻辑（不碰 LLM）
from typing import TypedDict

class RAGState(TypedDict):
    question: str
    documents: list
    rewrites: int

def decide(documents: list, rewrites: int, max_rewrites: int = 2):
    """rag_qa_project 的路由函数（简化版，纯逻辑可直接测）"""
    if documents:
        return "generate"
    if rewrites < max_rewrites:
        return "rewrite"
    return "generate"

# 三条路径全覆盖
assert decide(["d1"], 0) == "generate"       # 有文档 -> 生成
assert decide([], 0) == "rewrite"            # 无文档且可重写
assert decide([], 2) == "generate"           # 重写耗尽 -> 兜底
print("路由逻辑测试全部通过（0 额度）")

路由逻辑测试全部通过（0 额度）


In [4]:

# 集成测试示范：在 notebook 里直接跑 rag_qa_project 的离线测试套件
import subprocess, sys
from pathlib import Path

proj = ROOT / "advanced_tutorial" / "rag_qa_project"
r = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=line"],
    cwd=proj, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-1500:])

============================= test session starts =============================
platform win32 -- Python 3.14.5, pytest-9.1.1, pluggy-1.6.0 -- D:\PycharmProjects\my_langchain_demo\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: D:\PycharmProjects\my_langchain_demo
configfile: pyproject.toml
plugins: anyio-4.14.2, langsmith-0.10.17
collecting ... collected 4 items

tests\test_graph.py::test_graph_compiles PASSED                          [ 25%]
tests\test_graph.py::test_happy_path PASSED                              [ 50%]
tests\test_graph.py::test_rewrite_loop_and_fallback PASSED               [ 75%]
tests\test_graph.py::test_generate_with_empty_documents PASSED           [100%]

============================== 4 passed in 2.60s ==============================



## 4. 性能优化

### 4.1 并行是第一杠杆

| 手段 | 语义 | 收益 |
|---|---|---|
| `RunnableParallel` | 结构性并行 | 分支耗时取 max 而非 sum |
| `.batch([...])` | 数据级并行 | 逐条→并发，内置节流 |
| `Send`（教程二） | 图内动态并行 | Map-Reduce 场景 |

In [ ]:

import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

qa = ChatPromptTemplate.from_template("一句话回答：{q}") | model | StrOutputParser()
questions = [{"q": q} for q in ["什么是RAG", "什么是Agent", "什么是Embedding"]]

# 串行
t0 = time.time()
serial = [qa.invoke(q) for q in questions]
t_serial = time.time() - t0

# batch 并发（内部线程池，自动节流）
t0 = time.time()
batched = qa.batch(questions)
t_batch = time.time() - t0

print(f"串行 {t_serial:.1f}s | batch {t_batch:.1f}s | 加速比 {t_serial/t_batch:.1f}x")

### 4.2 上下文裁剪：trim_messages

多轮对话越聊越长，token 成本线性膨胀。
`trim_messages` 按 token 上限裁剪历史（优先保留 system + 最新消息）：

In [15]:

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, trim_messages

history = [SystemMessage(content="你是助手")] + [
    m
    for i in range(1, 8)
    for m in (HumanMessage(content=f"第{i}个问题：{i}+{i}=?"),
              AIMessage(content=f"等于 {i*2}"))
]

trimmed = trim_messages(
    history,
    max_tokens=10,
    strategy="last",                        # 保留最近的消息
    token_counter=len,                      # 演示用字符数当 token 数
    include_system=True,                    # system 永不裁
    start_on="human",                       # 从 HumanMessage 开始（成对裁剪）
)
print(f"裁剪前 {len(history)} 条 -> 裁剪后 {len(trimmed)} 条")
print("首条:", trimmed[0].content[:20], "| 末条:", trimmed[-1].content)

裁剪前 15 条 -> 裁剪后 9 条
首条: 你是助手 | 末条: 等于 14


### 4.3 优化清单（按性价比排序）

1. **并行化**（batch / Parallel / Send）——不改质量纯提速
2. **上下文裁剪**（trim_messages）——直接省 token 钱
3. **小模型分工**——路由/评分用轻量档，只有生成用旗舰（rag_qa_project 的 grade 就适合）
4. **缓存**——`set_llm_cache(SQLiteCache(...))` 缓存重复问题；嵌入用 CacheBackedEmbeddings
5. **温度调低**——RAG 类任务 0.1 既省心又减少重试
6. **recursion_limit 设防线**——循环图失控 = 烧钱事故，务必显式设置

## 5. 常见问题（FAQ）

| 问题 | 原因 | 解决 |
|---|---|---|
| 测试里 with_structured_output 报错 | Fake 模型未实现 | 参考 rag_qa_project/tests 的覆写写法 |
| subprocess 跑 pytest 中文乱码 | Windows GBK 控制台 | encoding="utf-8", errors="replace" |
| batch 没提速 | 目标服务限流 | 并发是本地开销的并行；服务端限流时收益有限 |
| trim 后消息不成对 | 未设 start_on | start_on="human" 保持对话完整性 |
| 改了 config 不生效 | .env 缓存/pydantic 实例早创建 | 重新实例化 Settings；检查 env_prefix |
| notebook 一切正常，脚本跑不了 | 工作目录/路径假设不同 | 用绝对路径 + Path(__file__) 锚定（见 run.py） |